# 00 · Kurulum ve yapılandırma
**TrustFed-TinyIDS** · FİNAL AŞAMA (Aşama 1–4 Colab'da yeniden; ardından F1, F2, F3)

Bu defter Google Drive proje klasörünü hazırlar, tüm defterlerin kullandığı **tek yapılandırma dosyasını**
yazar ve TON_IoT veri dosyasını doğrular.

**Drive klasör yapısı** (`/content/drive/MyDrive/TrustFed-TinyIDS-Final/`):
```
trustids/        ortak kod (proje klasöründen yükleyin)
notebooks/       bu defterler
data/            train_test_network.csv  (siz ekleyeceksiniz)
configs/         default.json            (burada yazılır)
artifacts/       hazırlanmış veri, skorlar, modeller (01–04 tarafından yazılır)
figures/, runs/  şekiller ve sonuç tabloları
```

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS-Final")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
    cfg.lofo_seeds, cfg.fl_orig_rounds = cfg.lofo_seeds[:2], 3
    cfg.resolution_cal_windows, cfg.resolution_eval_windows = 600, 600
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

## 1 · Klasörleri oluştur ve yapılandırmayı kaydet
Önceden kaydedilmiş (pre-registered) tüm seçimler — görülmeyecek saldırı aileleri, α, β, bölmeler, rastgele tohumlar — `configs/default.json` içindedir. **Test sonuçlarına baktıktan sonra değiştirilmez.**

In [ ]:
for d in ["data", "artifacts", "runs", "configs", "figures"]:
    (Path(PROJECT_DIR) / d).mkdir(parents=True, exist_ok=True)
import json, datetime
from dataclasses import asdict
current = {**asdict(Config()), **({} if CONFIG_NAME == "default.json" else asdict(cfg))}
current["project_dir"] = PROJECT_DIR
if not CFG_PATH.exists():
    Config(**current).save(CFG_PATH)      # hızlı-mod değişiklikleri değil, önceden kaydedilmiş değerler yazılır
    print("Yapılandırma yazıldı:", CFG_PATH)
else:
    old = json.loads(CFG_PATH.read_text(encoding="utf-8"))
    diff = {k: (old.get(k), v) for k, v in current.items() if old.get(k) != v}
    if diff and CONFIG_NAME == "default.json":
        # eski bir Colab çalıştırmasından kalan yapılandırma, yeni kodla sessizce karışmasın
        backup = CFG_PATH.with_name(f"default.{datetime.datetime.now():%Y%m%d-%H%M%S}.bak.json")
        backup.write_text(json.dumps(old, indent=2), encoding="utf-8")
        Config(**current).save(CFG_PATH)
        cfg = Config.load(CFG_PATH)
        print("Drive'daki yapılandırma koddan farklıydı; yedeklendi ve güncellendi:", backup.name)
        for k, (a, b) in diff.items():
            print(f"  {k}: {a} -> {b}")
    else:
        print("Yapılandırma güncel:", CFG_PATH)
print("Görülmeyecek (held-out) aileler — son değerlendirmeye kadar hiç kullanılmaz:", cfg.heldout_families)
print("Geliştirme (bilinen) aileleri:", cfg.dev_families)
print("alpha =", cfg.alpha, "| beta =", cfg.beta)

## 2 · TON_IoT ağ dosyasını edinin
Öneri metni **sürüm A**'yı birincil kabul eder (UNSW resmî `train_test_network.csv`, 461.043 satır, `ts` sütunu var).

1. Resmî TON_IoT sayfasından indirin (UNSW Canberra, *Train_Test_datasets → Train_Test_Network_dataset*).
   SharePoint bağlantısı genellikle `wget` ile çalışmaz; tarayıcıdan indirin.
2. `MyDrive/TrustFed-TinyIDS-Final/data/train_test_network.csv` konumuna yükleyin.

Yalnızca ayna (mirror) kopyası varsa (ör. Hugging Face `codymlewis/TON_IoT_network`, **sürüm B**: 211.043 satır,
`ts` yok), onu kullanın; ancak raporda bu belirtilmeli ve zamansal analiz (S4) çıkarılmalıdır.

In [ ]:
from trustids import data as D
path = cfg.data_path
assert path.exists(), f"Veri dosyası bulunamadı: {path}. Önce Drive'a yükleyin (yukarıdaki talimatlar)."
raw = D.load_raw(path, cfg)
info = {**D.detect_version(raw), "sha256": utils.sha256_file(path), "file": str(path)}
utils.save_json(info, ART / "data_file_info.json")
info

In [ ]:
env = utils.environment_info()
utils.save_json(env, ART / "environment.json")
env

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")